In [1]:
from IPython.display import Video
import cv2
from PIL import Image
import numpy as np
import ffmpeg
import os
import json
import hashlib
import subprocess
from datetime import datetime
from pathlib import Path
import exiftool
import re

In [2]:
def process_video(
    input_path: str,
    output_path: str,
    target_width: int,
    target_height: int,
    start_time: float | None = None,
    duration: float | None = None,
    fps: int | float | None = None,
) -> None:
    """
    adjust resolution, FPS, start offset, and duration

    :param input_path: Path to the source video file.
    :param output_path: Path for the exported video file.
    :param target_width: Target canvas width in pixels.
    :param target_height: Target canvas height in pixels.
    :param start_time: Start offset in seconds (optional).
    :param duration: Total length to extract in seconds from start_time (optional).
    :param fps: Desired output frame rate (optional).
    """
    input_kwargs = {}

    if start_time is not None:
        input_kwargs["ss"] = start_time

    if duration is not None:
        input_kwargs["t"] = duration

    stream = ffmpeg.input(input_path, **input_kwargs)

    # separate video stream
    video = stream.video

    # scale maintaining aspect ratio (black bars)
    video = video.filter(
        "scale",
        w=target_width,
        h=target_height,
        force_original_aspect_ratio="decrease",
    ).filter(
        "pad",
        w=target_width,
        h=target_height,
        x="(ow-iw)/2",
        y="(oh-ih)/2",
        color="black",
    )

    # apply frame rate filter if specified
    if fps is not None:
        video = video.filter("fps", fps=fps)

    output_kwargs = {
        "vcodec": "libx264",
        "acodec": "aac",
        "pix_fmt": "yuv420p",
    }

    output = ffmpeg.output(video, output_path, **output_kwargs)

    # execute
    ffmpeg.run(output, overwrite_output=True)


In [3]:
def extract_datetime_from_filename(filename: str) -> str | None:
    """Extracts timestamp from DJI pattern: DJI_YYYYMMDDHHMMSS_..."""
    match = re.search(r"(\d{4})(\d{2})(\d{2})(\d{2})(\d{2})(\d{2})", filename)
    if match:
        year, month, day, hour, minute, second = match.groups()
        return f"{year}{month}{day}_{hour}{minute}"
    return None


def get_recording_datetime_and_metadata(file_path: Path) -> tuple[str, dict]:
    """extracts metadata using ExifTool, fallback regex, and FFprobe."""
    resolved_path = str(file_path.resolve())
    stat = file_path.stat()
    
    metadata = {
        "file_system": {
            "filename": file_path.name,
            "original_path": resolved_path,
            "size_bytes": stat.st_size,
            "filesystem_mtime": datetime.fromtimestamp(stat.st_mtime).isoformat(),
            "extension": file_path.suffix.lower()
        },
        "exif_metadata": {},
        "ffprobe_streams": {}
    }

    # 1. Read metadata (ExifTool)
    try:
        with exiftool.ExifToolHelper() as et:
            raw_exif = et.get_metadata(resolved_path)
            if raw_exif:
                metadata["exif_metadata"] = raw_exif[0]
    except Exception as e:
        metadata["exif_metadata"] = {"error": str(e)}

    # read streams (FFprobe)
    cmd = [
        "ffprobe",
        "-v", "quiet",
        "-print_format", "json",
        "-show_format",
        "-show_streams",
        resolved_path
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        metadata["ffprobe_streams"] = json.loads(result.stdout)
    except Exception as e:
        metadata["ffprobe_streams"] = {"error": str(e)}

    # true recording timestamp
    exif = metadata["exif_metadata"]
    date_str = None
    
    # Check EXIF/QuickTime internal date tags
    candidate_keys = [
        "QuickTime:CreationDate",
        "QuickTime:CreateDate",
        "EXIF:DateTimeOriginal",
        "XMP:CreateDate",
        "XMP:DateTimeOriginal"
    ]
    
    for key in candidate_keys:
        raw_val = exif.get(key)
        if raw_val and isinstance(raw_val, str) and not raw_val.startswith("0000"):
            # "2025:11:03 10:53:07" or "2025:11:03 10:53:07-04:00"
            clean_date = raw_val.split("+")[0].split("-")[0].strip() if "+" in raw_val or ("-" in raw_val and raw_val.count("-") == 1) else raw_val
            clean_date = re.sub(r"[^\d]", "", clean_date)[:14] # YYYYMMDDhhmmss
            if len(clean_date) == 14:
                date_str = f"{clean_date[:8]}_{clean_date[8:]}"
                break

    if not date_str: # extract from filename
        date_str = extract_datetime_from_filename(file_path.name)

    if not date_str: # modified time on disk
        date_str = datetime.fromtimestamp(stat.st_mtime).strftime("%Y%m%d_%H%M%S")

    metadata["recording_timestamp"] = date_str
    return date_str, metadata


def generate_unique_folder_name(file_path: Path, recording_timestamp: str) -> str:
    """recording timestamp with short hash"""
    hash_str = hashlib.sha256(str(file_path.resolve()).encode("utf-8")).hexdigest()[:8]
    return f"{recording_timestamp}_{hash_str}"

def process_video_dataset(input_dir: str, output_base_dir: str) -> None:
    """recursively traverses input_dir, ignores non-videos, extracts real metadata, and runs process_video."""
    
    input_folder = Path(input_dir)
    output_folder = Path(output_base_dir)

    valid_extensions = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".ts"}
    
    # search all subdirectories
    video_files = [
        f for f in input_folder.rglob("*") 
        if f.is_file() and f.suffix.lower() in valid_extensions
    ]

    print(f"Found {len(video_files)} video(s) to process.")

    for video_path in video_files:
        # metadata
        recording_date, full_metadata = get_recording_datetime_and_metadata(video_path)

        # output directory
        unique_folder_name = generate_unique_folder_name(video_path, recording_date)
        dest_dir = output_folder / unique_folder_name
        dest_dir.mkdir(parents=True, exist_ok=True)

        # output paths
        output_video_path = dest_dir / f"video.mp4"
        metadata_path = dest_dir / "metadata.json"

        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(full_metadata, f, indent=4, ensure_ascii=False)

        print(f"\n[+] Processing: {video_path.relative_to(input_folder)}")
        print(f"    Recorded on: {recording_date} -> Output: {dest_dir.name}")
        
        process_video(
            input_path=str(video_path),
            output_path=str(output_video_path),
            target_width=1024,
            target_height=576,
            start_time=0.0,
            fps=15
        )

In [4]:
process_video_dataset("./dataset/unprocessed", "./dataset/processed")

Found 8 video(s) to process.

[+] Processing: DJI_0720.MP4
    Recorded on: 20260526_133321 -> Output: 20260526_133321_48134879


ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml


[+] Processing: .ipynb_checkpoints/DJI_0720-checkpoint.MP4
    Recorded on: 20260526_133321 -> Output: 20260526_133321_c0d252ef


ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml


[+] Processing: myairbridge-Vt1EbS3JhOo/DJI_20251215180333_0089_D - Delfines chilenos.MP4
    Recorded on: 20260813_233250 -> Output: 20260813_233250_ee3d47e3


...

ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml


[+] Processing: myairbridge-Vt1EbS3JhOo/DJI_20251103105307_0001_D - Delfines oscuros y Lisos 2.mp4
    Recorded on: 20260814_004529 -> Output: 20260814_004529_01db9042


ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml


[+] Processing: myairbridge-Vt1EbS3JhOo/DJI_0005 - Delfines chilenos.MOV
    Recorded on: 20260813_233811 -> Output: 20260813_233811_b384f7f9


ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml